# 1. Data loading
First of all, we import all libraries we used (made at the end) along the whole proyect to have a more readable code.

We load the data and show basic information (rows, columns, types).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier



df = pd.read_csv("Road Accident Data.csv")
df.head()
df.info()
df.shape

#  2. Initial exploration of the dataset

In this section we analyze:
- Number of rows and columns
- Data type
- Null values
- Distribution of goal variable

In [ ]:
df.isnull().sum().sort_values(ascending=False)
df['Accident_Severity'].value_counts(normalize=True)

#  3. Cleaning and preprocessing the dataset

What we did here:


*  Correction of an incorrect category label ("Fetal")
*  Removal of irrelevant columns (the ones we have a 100% we won't use)
*  Imputation of missing values in categorical features
*   Date feature processing
*   Accident time processing
*   Frequency encoding for high-cardinality categorical features
*   Transformation of the target variable into numerical values (0, 1, 2)


All preprocessing steps were applied using the preprocess_accidents() function to ensure a clean and reproducible pipeline.

In [ ]:
def preprocess_accidents(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["Accident_Severity"] = df["Accident_Severity"].replace("Fetal", "Fatal")

    cols_to_drop = []
    if "Accident_Index" in df.columns:
        cols_to_drop.append("Accident_Index")
    if "Carriageway_Hazards" in df.columns:
        cols_to_drop.append("Carriageway_Hazards")
    df = df.drop(columns=cols_to_drop)

    # This preserves the pattern of "missingness" which can sometimes be predictive.
    fill_unknown = ["Weather_Conditions", "Road_Type", "Road_Surface_Conditions", "Time"]
    for col in fill_unknown:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")

    # Temporal Feature Engineering: Extracting cyclical components (Month/Day)
    if "Accident Date" in df.columns:
        df["Accident_Date_dt"] = pd.to_datetime(df["Accident Date"], errors="coerce")
        df["Accident_Year"] = df["Accident_Date_dt"].dt.year
        df["Accident_Month"] = df["Accident_Date_dt"].dt.month
        df["Accident_DayOfYear"] = df["Accident_Date_dt"].dt.dayofyear
        df = df.drop(columns=["Accident Date", "Accident_Date_dt"])

    # Time-based Feature Engineering: Creating domain-relevant flags
    # 'Is_Rush_Hour' captures traffic density patterns which are correlated with accident types.
    if "Time" in df.columns:
        time_dt = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce")
        df["Hour"] = time_dt.dt.hour
        df["Is_Night"] = df["Hour"].isin([0,1,2,3,4,5,22,23]).astype(int)
        df["Is_Rush_Hour"] = (
            df["Hour"].between(7,9).astype(int) |
            df["Hour"].between(16,19).astype(int)
        ).astype(int)
        df = df.drop(columns=["Time"])

    high_card = ["Local_Authority_(District)", "Police_Force"]
    for col in high_card:
        freq = df[col].value_counts(normalize=True)
        df[col + "_freq"] = df[col].map(freq)
        # df = df.drop(columns=[col])


    # Mapping Target Variable to Ordinal Integers
    map_sev = {"Slight": 0, "Serious": 1, "Fatal": 2}
    df["Accident_Severity_num"] = df["Accident_Severity"].map(map_sev)

    return df

df = preprocess_accidents(df)
df.head()


#  4. Compact univariate EDA

Instead of generating dozens of plots, we use a compact table that summarizes:



*   Variable type
*   Number of unique values
*   Percentage of missing values
*   Basic statistics (numerical variables)
*   Most frequent category (categorical variables)


In [ ]:
def resumen_univariante_compacto(df):
    resumen = []
    for col in df.columns:
        tipo = df[col].dtype
        n_unique = df[col].nunique()
        n_null = df[col].isna().sum()
        null_pct = n_null / len(df) * 100

        if tipo != "object":
            desc = df[col].describe()
            resumen.append({
                "column": col,
                "type": "numeric",
                "n_unique": n_unique,
                "nulls_%": round(null_pct,2),
                "min": desc["min"],
                "max": desc["max"],
                "mean": round(desc["mean"],3),
                "std": round(desc["std"],3),
            })
        else:
            top = df[col].value_counts(normalize=True).head(1)
            resumen.append({
                "column": col,
                "type": "categoric",
                "n_unique": n_unique,
                "nulls_%": round(null_pct,2),
                "most frequent category": top.index[0],
                "%_más_frecuente": round(top.iloc[0]*100,2),
            })

    return pd.DataFrame(resumen)

resumen_univariante_compacto(df)


#  5. Bivariate EDA: importance ranking

For each variable, we measure:


*   Absolute correlation (numerical variables)
*   Severity variance by category (categorical variables)

This produces an objective ranking of which variables are possible high indicators of accident severity.

In [ ]:
def variable_severity_score(df, cols, target="Accident_Severity_num"):
    scores = {}
    for col in cols:
        if df[col].dtype == "object":
            score = df.groupby(col)[target].mean().var()
        else:
            score = abs(df[col].corr(df[target]))
        scores[col] = score
    return pd.Series(scores).sort_values(ascending=False)

cols = [c for c in df.columns if c != "Accident_Severity_num"]
ranking = variable_severity_score(df, cols)
ranking


#  6. Final variable selection

We take the variabless with evidence of predictive value, even if it is low:

- Speed_limit  
- Number_of_Vehicles  
- Number_of_Casualties (*)  
- Is_Night  
- Is_Rush_Hour  
- Local_Authority_freq  
- Police_Force_freq  
- Latitude  
- Longitude
- Accident day of the year
- Accident month
- Accident hour
- Type of vehicle in the accident
- Conditions of the road where the accident occurred
- Light conditions
- Weather conditions
- Road type on the accident place
- Type of area in the accident; urban or rural
- Type of junction
- Type of control in the junction
- Day of the week when the accident occurred



(*) Note: `Number_of_Casualties` is post-accident, it should be used when the goal is analysis, not real time prediction.

In [ ]:
selected = [
    # CRITICAL: We exclude 'Number_of_Casualties' because it is a post-accident feature.
    # Including it would cause Data Leakage, as the number of casualties is a direct outcome of severity.
    "Speed_limit",
    "Number_of_Vehicles",
    "Is_Night",
    "Is_Rush_Hour",
    "Local_Authority_(District)_freq",
    "Police_Force_freq",
    "Latitude",
    "Longitude",
    "Accident_DayOfYear",
    "Accident_Month",
    "Hour",
    "Vehicle_Type",
    "Road_Surface_Conditions",
    "Light_Conditions",
    "Weather_Conditions",
    "Road_Type",
    "Urban_or_Rural_Area",
    "Junction_Detail",
    "Junction_Control",
    "Day_of_Week",
]

target = "Accident_Severity_num"

df_model = df[selected + [target]].copy()

# One-Hot Encoding for Low-Cardinality Categoricals
df_model = pd.get_dummies(df_model, drop_first=True) # Transform categorical text data to numerical categorical data (f.e. days of the week) -> Necessary for Logistic Regression as it cannot handle string categories.
df_model = df_model.dropna() #Drop NAs for ensuring model stability.
df_model.head(), df_model.shape


#  7. Train/Test split (stratified)

The dataset is heavily imbalanced, so we use stratify=y
to ensure that class proportions are preserved in both sets.


In [ ]:
X = df_model.drop(columns=[target])
y = df_model[target]

# Stratify=y is essential here due to the severe class imbalance (Fatal ~1-2%).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train.shape, X_test.shape, y_train.value_counts(normalize=True)


#  8. Balancing classes

The “Fatal” class is ≈ 1%, so the model can’t learn properly without balancing techniques.

Methods applied:

### ✔ class_weight="balanced"
Recalculates the importance of each class given frequency.

In [ ]:
# We compute weights inversely proportional to class frequencies to penalize the model
# more heavily for misclassifying minority classes (Fatal/Serious).
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), weights))
class_weights

#  9. Baseline model: Logistic Regression with class balancing



The features are scaled with StandardScaler to standardize values. LogisticRegression is trained on the scaled training data with class_weight='balanced' to handle the imbalanced severity classes. Predictions on the test set are generated, and accuracy plus a classification report are printed.






In [ ]:
# Standardization (Z-score normalization) is required for Logistic Regression to ensure
# convergence and allow for correct interpretation of coefficients.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(class_weight=class_weights, max_iter=1000, random_state=42)

print("Training the model")
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("\n--- Model results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Slight", "Serious", "Fatal"]))

# 9.1 Confusion matrix-Logistic Regression (raw counts)
confusion_matrix compares predicted vs. true labels. sns.heatmap visualizes counts per class, showing where the model succeeds or misclassifies Slight, Serious, and Fatal accidents.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Slight", "Serious", "Fatal"],
    yticklabels=["Slight", "Serious", "Fatal"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – Logistic Regression")
plt.show()


# 9.2. More on Logistic Regression evaluation
The confusion matrix is normalized by row to show per-class accuracy as percentages. cross_val_score with StratifiedKFold evaluates F1 macro across folds to check robustness. roc_auc_score computes multiclass ROC AUC using predicted probabilities. Feature importance is extracted from model coefficients, highlighting which variables most influence severity predictions.

In [ ]:
# Normalized confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
cm_norm = cm / cm.sum(axis=1)[:, None]

class_names = ["Slight", "Serious", "Fatal"]

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix – Logistic Regression")
plt.show()

# Cross-validation (F1 macro)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f"Logistic Regression CV F1 Macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Multiclass ROC AUC
y_proba = model.predict_proba(X_test_scaled)
auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr')
print(f"Logistic Regression ROC AUC (OVR): {auc_score:.4f}")

# Coefficients and feature importance
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nTop 10 features by absolute coefficient:")
print(coef_df.head(10))


# 10.Random Forest model
Trains a Random Forest classifier to predict accident severity using only pre-accident features. This ensures the model learns patterns that could be known before an accident occurs, avoiding post-accident information that would artificially inflate performance. Class weights address the strong class imbalance, and the model is evaluated on a stratified test set with a classification report.

In [ ]:
# Note: Random Forest is a tree-based ensemble method. Unlike Logistic Regression,
# it is invariant to feature scaling, so raw X_train could be used.
# However, using the processed X_train maintains consistency in the pipeline.
X_train_rf = X_train
X_test_rf = X_test
y_train_rf = y_train
y_test_rf = y_test

# n_estimators=500: Sufficient trees to stabilize variance without excessive computational cost.
# class_weight=class_weights: Cost-sensitive learning to address the imbalance.
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=50, # Conservative pruning to prevent overfitting
    random_state=42,
    class_weight=class_weights,
    n_jobs
1
)

rf.fit(X_train_rf, y_train_rf)

y_pred_rf = rf.predict(X_test_rf)

# We focus on Recall for the minority class ('Fatal') rather than just global Accuracy.
print(f"\nAccuracy Random Forest: {accuracy_score(y_test_rf, y_pred_rf):.4f}")
print(classification_report(y_test_rf, y_pred_rf, digits=4))



# 10.1. Confusion matrix-Random Forest (raw counts)
Shows not normalized confusion matrix for the Random Forest predictions, highlighting how many accidents were correctly or incorrectly classified for each severity category. This helps visualize the model’s performance across all classes, including the rare cases.

In [ ]:
cm_rf = confusion_matrix(y_test_rf, y_pred_rf, labels=[0, 1, 2])

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_rf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Slight", "Serious", "Fatal"],
    yticklabels=["Slight", "Serious", "Fatal"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – Random Forest")
plt.show()



# 10.2. More on Random Forest evaluation
This cell evaluates the Random Forest model using multiple perspectives. A normalized confusion matrix shows how well the model predicts each severity class. Cross-validation with F1 macro score provides a robust estimate of model performance on unseen data. Multiclass ROC AUC assesses the model’s ability to distinguish between severity categories. Finally, feature importance highlights which pre-accident variables contribute most to the predictions.


In [ ]:
# Normalized confusion matrix
cm_rf = confusion_matrix(y_test_rf, y_pred_rf, labels=[0, 1, 2])
cm_norm_rf = cm_rf / cm_rf.sum(axis=1)[:, None]

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_norm_rf,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=["Slight", "Serious", "Fatal"],
    yticklabels=["Slight", "Serious", "Fatal"]
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix – Random Forest")
plt.show()

# C01ross-validation (F1 macro)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train_rf, y_train_rf, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f"Random Forest CV F1 Macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Multiclass ROC AUC
if hasattr(rf, "predict_proba"):
    y_proba_rf = rf.predict_proba(X_test_rf)
    auc_score = roc_auc_score(y_test_rf, y_proba_rf, multi_class='ovr')
    print(f"Random Forest ROC AUC (OVR): {auc_score:.4f}")

# Feature importance
feature_importances = pd.DataFrame({
    "Feature": X_train_rf.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop 10 features by importance:")
print(feature_importances.head(10))
